# Level 3 - SENIOR test
**Rules:** about 2 hours. No peeking. For the written questions, set a timer for **8 minutes each**, write your answer in the markdown cell,
*then* score it with the rubric in `levels/ANSWER_KEY.md`. Pass mark: **85%**.

| Section | Items |
|---|---|
| Coding | S1-S5 |
| Spark | SS1-SS3 |
| Multiple choice | Q1-Q6 |
| Written scenarios | W1-W5 (self-scored 0-5) |

In [ ]:
import os, sys
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, REPO_ROOT)
from pyspark.sql import Window, functions as F
from levels.grader import Scorecard

card = Scorecard("senior", spark)
events, change_log, orders = card.data["events"], card.data["change_log"], card.data["orders"]

## Coding
### S1 - Network delay time
`times` is a list of `[u, v, w]` (a directed edge u -> v that takes w >= 0). Nodes are `1..n`. A signal starts at `k`. Return the time until **all** nodes
have received it, or `-1` if some never do.

In [ ]:
def network_delay_time(times, n, k):
    raise NotImplementedError

card.grade("S1", network_delay_time)

### S2 - LRU cache
`LRUCache(capacity)` with `get(key)` (returns the value or `-1`) and `put(key, value)`, both **O(1)**. Both `get` and `put` (including updates) count as a use.
When over capacity, evict the least recently used key.

In [ ]:
class LRUCache:
    def __init__(self, capacity):
        raise NotImplementedError

    def get(self, key):
        raise NotImplementedError

    def put(self, key, value):
        raise NotImplementedError

card.grade("S2", LRUCache)

### S3 - Overlapping validity windows
`windows` is a list of `(key, start, end)` where windows are **half-open** `[start, end)` and `end=None` means open-ended. Input is unsorted.
Return a **sorted list of keys** that have at least one pair of overlapping windows.

In [ ]:
def overlapping_keys(windows):
    raise NotImplementedError

card.grade("S3", overlapping_keys)

### S4 - Minimum concurrent slots
Given half-open job intervals `(start, end)`, return the minimum number of slots needed so that no two jobs overlap in the same slot.
Target: O(n log n).

In [ ]:
def min_concurrent(intervals):
    raise NotImplementedError

card.grade("S4", min_concurrent)

### S5 - Entity resolution
`records` is a dict `record_id -> list of identifiers`. Records sharing any identifier belong to the same entity (transitively).
Return the groups as a list of **sorted** lists of record ids, with the outer list sorted too. Records with no identifiers are their own group.

In [ ]:
def resolve_entities(records):
    raise NotImplementedError

card.grade("S5", resolve_entities)

## Spark
### SS1 - Sessionize clickstream
`events(user_id, ts)`. A new session starts at a user's first event, or when the gap since their previous event is **more than** 30 minutes (exactly 30 minutes is still the same session).
Columns: `user_id`, `session_id` (1, 2, 3... per user in time order), `session_start`, `session_end`, `n_events`.

In [ ]:
ss1 = None

card.grade_df("SS1", ss1)

### SS2 - Rebuild SCD2 from an out-of-order change log
`change_log(customer_id, region, effective_date, ingested_at)`. Rules:
- several rows for the same (customer, effective_date): the **latest `ingested_at`** wins
- a row whose region equals the **previous version's** region (by effective_date) is not a change, so drop it
- `valid_to` = the next version's `valid_from` (exclusive), NULL for the latest; `is_current` = (`valid_to` is NULL)

Columns: `customer_id`, `region`, `valid_from`, `valid_to`, `is_current`.

In [ ]:
ss2 = None

card.grade_df("SS2", ss2)

### SS3 - Longest ordering streak
`orders(customer_id, order_date)`, which can have several orders on the same day. For each customer, return the length of the longest run of **consecutive calendar days** with at least one order.
Columns: `customer_id`, `longest_streak`.

In [ ]:
ss3 = None

card.grade_df("SS3", ss3)

## Multiple choice
**Q1.** AQE's skew-join optimization works by: a) salting keys automatically b) splitting oversized partitions into several tasks, each joined with the matching
partition from the other side c) broadcasting the large side d) dropping skewed keys

**Q2.** Salting a join key into N buckets costs you: a) a fact side duplicated N times b) nothing c) the other (dimension) side replicated N times d) an extra broadcast of the fact table

**Q3.** Primary/foreign key constraints on Unity Catalog tables are: a) enforced on every write b) disabled on Delta c) only allowed on views
d) informational, and with `RELY` the optimizer may use them for rewrites

**Q4.** To make a streaming `foreachBatch` upsert safe to retry: a) MERGE on the business key, and/or pass `txnAppId` + `txnVersion` b) use append mode
c) increase the trigger interval d) delete the checkpoint before each run

**Q5.** `SELECT count(*) FROM delta_table` is often very fast because: a) the result is always cached b) it samples the data
c) row counts can come from file statistics in the Delta transaction log d) Photon skips counting

**Q6.** A BI dashboard repeatedly runs the same heavy aggregation over a 20 TB fact table. The best first move: a) serve it from a materialized view or gold
aggregate table refreshed incrementally b) the largest SQL warehouse size c) disable Photon d) export to CSV for the BI tool

In [ ]:
answers = {"Q1": "", "Q2": "", "Q3": "", "Q4": "", "Q5": "", "Q6": ""}
for q, a in answers.items():
    card.mcq(q, a)

## Written scenarios (8 minutes each, then self-score 0-5 using `levels/ANSWER_KEY.md`)
Use the framework: clarify -> measure/current state -> options -> recommendation -> risks/validation -> metrics.

**W1 - Diagnosis.** A stage shows: 400 tasks, median 6 s, max 41 min; the slow task has 38 GB shuffle read and 52 GB spill; the join key is
`customer_id`; the dimension is 3 GB. What's happening, what do you check next, and what do you change?

*Your answer:*

**W2 - Warehouse design.** A retailer wants daily sales, inventory, and customer-segment reporting for 300 stores, with history of customer
segment changes, plus near-real-time online orders. Design the model (facts, grain, dimensions, SCD types) and the pipeline on Databricks.

*Your answer:*

**W3 - Migration.** A customer has 3,000 Teradata BTEQ scripts, 40 TB, and 600 Tableau dashboards, and their contract ends in 9 months. Plan the migration.

*Your answer:*

**W4 - Trade-off.** A 50 TB events table is queried by `event_date` (always) and often by `user_id`. The data engineer wants
`PARTITIONED BY (event_date, user_id)`. Respond.

*Your answer:*

**W5 - Customer push-back.** The VP of Analytics says: "Our dashboards were faster on our old warehouse. Databricks SQL is slow." What do you do in the
first two weeks?

*Your answer:*

In [ ]:
written_scores = {"W1": None, "W2": None, "W3": None, "W4": None, "W5": None}   # 0-5 each, from the rubric
for w, s in written_scores.items():
    card.self_score(w, s, out_of=5)

## Scorecard
Run this last. It totals every section, including your self-scored written answers, prints your percentage and a verdict, and lists the questions to review.

In [ ]:
card.summary()

**Review:** compare each written answer against its rubric in `levels/ANSWER_KEY.md`, and check `levels/reference.py` for the coding and Spark solutions. Rewrite your weakest written answer tomorrow, out loud, in 8 minutes.